## Seleniumbase 테스트
아마존 top10 상품 정보 크롤링

In [ ]:
# !pip install seleniumbase

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached websockets-16.0-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/659.3 kB ? eta -:--:--
   ---------------------------------------- 659.3/659.3 kB 24.8 MB/s  0:00:00
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
Using cached websockets-16.0-cp311-cp311-win_amd64.whl (178 kB)

  Attempting uninstall: websockets

    Found existing installation: websockets 15.0.1

    Uninstalling websockets-15.0.1:

      Successfully uninstalled websockets-15.0.1

   - --------------------------------------  1/25 [websockets]
  Attempting uninstall: setuptools
   - --------------------------------------  1/25 [websockets]
    Found existing installation: setuptools 80.10.2
   - --------------------------------------  1/25 [websockets]
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
open-deep-research 0.0.16 requires beautifulsoup4==4.13.3, but you have beautifulsoup4 4.14.3 which is incompatible.
realtime 2.28.0 requires websockets<16,>=11, but you have websockets 16.0 which is incompatible.


In [15]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import json
from datetime import datetime

def fetch_amazon_kbeauty_final_headless():
    options = uc.ChromeOptions()
    options.add_argument('--headless')
    
    # --- [강력 추천] 에러 방지용 필수 옵션 ---
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu') 
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--incognito') 

    # 실제 브라우저 User-Agent 재설정
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    options.add_argument(f'user-agent={user_agent}')

    driver = None
    try:
        driver = uc.Chrome(options=options)
        wait = WebDriverWait(driver, 30)

        print("🌐 Headless 환경 우회 접속 시도 중...")
        
        # 1. 아마존 메인 접속
        driver.get("https://www.amazon.com")
        time.sleep(random.uniform(3, 5))

        if "Amazon.com" not in driver.title:
            driver.refresh()
            time.sleep(5)

        # 2. 배송지 LA(90001) 설정
        print("📍 배송지 위치 변경 중...")
        address_btn = wait.until(EC.element_to_be_clickable((By.ID, "nav-global-location-popover-link")))
        address_btn.click()
        
        zip_input = wait.until(EC.visibility_of_element_located((By.ID, "GLUXZipUpdateInput")))
        zip_input.send_keys("90001")
        time.sleep(1)
        driver.find_element(By.ID, "GLUXZipUpdate").click()
        
        try:
            done_btn = wait.until(EC.element_to_be_clickable((By.NAME, "glowDoneButton")))
            done_btn.click()
        except:
            driver.refresh()
        time.sleep(3)

        # 3. K-Beauty 검색 결과 페이지로 이동 (인기순 정렬)
        print("📊 데이터 페이지로 이동 중...")
        search_url = "https://www.amazon.com/s?k=k-beauty&s=exact-aware-popularity-rank"
        driver.get(search_url)
        time.sleep(random.uniform(4, 6))

        # 4. 데이터 파싱
        print("📦 Top 10 데이터 추출 중 (Rating/Review 포함)...")
        soup = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.find_all("div", {"data-component-type": "s-search-result"})
        
        final_rankings = []
        for item in items:
            if len(final_rankings) >= 10:
                break
            try:
                asin = item.get('data-asin')
                if not asin: continue
                
                # A. 제목 및 URL
                h2_tag = item.select_one("h2.a-size-base-plus.a-text-normal") or item.select_one("h2")
                title = h2_tag.get('aria-label') or h2_tag.get_text(strip=True) if h2_tag else "N/A"
                
                parent_a = h2_tag.find_parent("a") if h2_tag else None
                product_url = "https://www.amazon.com" + parent_a.get('href').split('?')[0] if parent_a else "N/A"

                # B. 브랜드
                brand_tag = item.select_one(".s-line-clamp-1 .a-color-base") or item.select_one(".a-size-mini.a-color-base")
                brand = brand_tag.get_text(strip=True) if brand_tag else "N/A"

                # C. 가격
                price_tag = item.select_one(".a-price .a-offscreen")
                price = price_tag.get_text(strip=True) if price_tag else "Check Price"

                # --- [핵심 수정] 평점(Rating) 추출 ---
                # data-cy="reviews-block" 내부의 별점 텍스트 직접 추출
                rating_tag = item.select_one('span[aria-hidden="true"].a-size-small.a-color-base') or \
                             item.select_one(".a-icon-alt")
                rating = "0"
                if rating_tag:
                    rating_val = rating_tag.get_text(strip=True).split(' ')[0]
                    rating = rating_val if rating_val.replace('.', '').isdigit() else "0"

                # --- [핵심 수정] 리뷰 개수(Reviews) 추출 ---
                # s-underline-text 클래스 혹은 aria-label 속성에서 숫자 추출
                review_tag = item.select_one('a[aria-label*="ratings"]') or \
                             item.select_one("span.s-underline-text")
                
                reviews = "0"
                if review_tag:
                    raw_review = review_tag.get('aria-label') or review_tag.get_text(strip=True)
                    # (2.7K) 또는 2,775 ratings 에서 숫자 데이터만 정제
                    reviews = raw_review.split(' ')[0].replace('(', '').replace(')', '').replace(',', '')

                final_rankings.append({
                    "rank": len(final_rankings) + 1,
                    "brand": brand,
                    "title": title,
                    "rating": float(rating),
                    "reviews": reviews, # '2.7K' 또는 '2775' 형태 유지
                    "price": price,
                    "url": product_url,
                    "asin": asin,
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                print(f"✅ {len(final_rankings)}위: [{brand[:10]}] {rating}⭐ ({reviews} reviews)")
            except Exception as e:
                continue

        # 5. JSON 저장
        filename = f"amazon_top10_final_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(final_rankings, f, ensure_ascii=False, indent=4)
        
        print(f"✨ 수집 성공! 파일명: {filename}")
        return final_rankings

    except Exception as e:
        print(f"❌ 최종 오류 발생: {e}")
        if driver:
            driver.save_screenshot("error_screenshot.png")
    finally:
        if driver:
            driver.quit()

if __name__ == "__main__":
    fetch_amazon_kbeauty_final_headless()

🌐 Headless 환경 우회 접속 시도 중...
📍 배송지 위치 변경 중...
📊 데이터 페이지로 이동 중...
📦 Top 10 데이터 추출 중 (Rating/Review 포함)...
✅ 1위: [AXIS-Y] 4.4⭐ (2775 reviews)
✅ 2위: [MEDITHERAP] 4.5⭐ (3897 reviews)
✅ 3위: [Touch in S] 4.3⭐ (13139 reviews)
✅ 4위: [Torriden] 4.5⭐ (3453 reviews)
✅ 5위: [Elizavecca] 4.6⭐ (4879 reviews)
✅ 6위: [LANEIGE] 4.2⭐ (218 reviews)
✅ 7위: [DERMAL] 4.6⭐ (25855 reviews)
✅ 8위: [N/A] 4.6⭐ (1599 reviews)
✅ 9위: [NICKA K NE] 4.5⭐ (15886 reviews)
✅ 10위: [SeoulCeuti] 4.3⭐ (27161 reviews)
✨ 수집 성공! 파일명: amazon_top10_final_20260315_213817.json


In [6]:
from seleniumbase import Driver
import time
import random
import json
from datetime import datetime
from bs4 import BeautifulSoup

def fetch_qoo10_kbeauty_top10_json():
    # 1. Driver 설정 (UC 모드 + Headless 활성화)
    # 큐텐도 봇 감지가 있으므로 UC 모드를 사용합니다.
    driver = Driver(uc=True, browser="chrome", headless=True)
    driver.set_window_size(1920, 1080)

    # 분석하신 Ajax 검색 결과 URL (리뷰 많은 순 정렬)
    target_url = "https://www.qoo10.jp/gmkt.inc/Search/SearchResultAjaxTemplate.aspx?keyword_hist=k-beauty&sortType=MOST_REVIEWED&pageSize=40&curPage=1"

    try:
        print("📡 Qoo10 Ajax 데이터 로드 중 (Headless)...")
        driver.uc_open_with_reconnect(target_url, reconnect_time=5)
        time.sleep(random.uniform(3, 5)) # 동적 로딩 대기

        # 2. BeautifulSoup으로 데이터 파싱
        print("📦 데이터 추출 시작...")
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # ga-product="goods" 속성을 가진 모든 행(tr)을 가져옵니다.
        items = soup.find_all("tr", {"ga-product": "goods"})
        
        final_rankings = []
        for item in items:
            if len(final_rankings) >= 10: # Top 10 제한
                break
                
            try:
                # A. 상품 ID (Goodscode)
                goods_code = item.get('goodscode')

                # B. 브랜드명 (Brand)
                brand_tag = item.select_one("a.txt_brand")
                brand = brand_tag.get_text(strip=True) if brand_tag else "N/A"

                # C. 상품명 (Title) 및 URL
                # sbj 클래스 안의 a 태그 중 브랜드 링크가 아닌 것 선택
                title_link = item.select_one(".td_item .sbj a:not(.txt_brand)")
                if title_link:
                    title = title_link.get('title') or title_link.get_text(strip=True)
                    product_url = title_link.get('href')
                else:
                    title = "N/A"
                    product_url = "N/A"

                # D. 가격 (Price)
                price_tag = item.select_one(".td_prc .prc strong")
                price = price_tag.get_text(strip=True) if price_tag else "N/A"

                # E. 리뷰 개수 (Reviews)
                review_tag = item.select_one(".review_total_count")
                reviews = review_tag.get_text(strip=True).replace('(', '').replace(')', '').replace(',', '') if review_tag else "0"

                final_rankings.append({
                    "rank": len(final_rankings) + 1,
                    "brand": brand,
                    "title": title,
                    "price": price,
                    "reviews": int(reviews) if reviews.isdigit() else 0,
                    "url": product_url,
                    "goods_code": goods_code,
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                print(f"✅ {len(final_rankings)}위 추출: [{brand}] {title[:25]}...")

            except Exception:
                continue

        # 3. JSON 파일 저장
        filename = f"qoo10_kbeauty_top10_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(final_rankings, f, ensure_ascii=False, indent=4)
        
        print(f"\n✨ 수집 및 저장 완료: {filename}")
        return final_rankings

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_qoo10_kbeauty_top10_json()

📡 Qoo10 Ajax 데이터 로드 중 (Headless)...
📦 데이터 추출 시작...

✨ 수집 및 저장 완료: qoo10_kbeauty_top10_20260315_212347.json


In [19]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import random
import json
from datetime import datetime

def fetch_qoo10_kbeauty_official_link():
    # 1. 드라이버 설정 (눈으로 확인하기 위해 Headless False)
    options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--incognito')

    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    options.add_argument(f'user-agent={user_agent}')

    driver = None
    try:
        driver = uc.Chrome(options=options)
        
        # 사용자님이 제공해주신 사이트 자체 검색 링크
        target_url = "https://www.qoo10.jp/s/?keyword_hist=k-beauty&sortType=MOST_REVIEWED&dispType=LIST&curPage=1"

        print(f"📡 큐텐 공식 검색 페이지 접속 중...")
        driver.get(target_url)
        
        # 큐텐은 리스트 로딩에 시간이 걸릴 수 있으므로 넉넉히 대기
        time.sleep(random.uniform(5, 8))

        # 2. 데이터 파싱 시작
        print("📦 페이지 로드 완료, 데이터 추출 시작...")
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # 제공해주신 HTML 구조에 따라 goods 또는 goods_power가 포함된 tr 선택
        items = soup.select('tr[ga-product^="goods"]')
        
        final_rankings = []
        for item in items:
            if len(final_rankings) >= 10: # 상위 10개만 수집
                break
                
            try:
                # A. 상품 코드
                goods_code = item.get('goodscode')

                # B. 브랜드명 (txt_brand 클래스)
                brand_tag = item.select_one("a.txt_brand")
                brand = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"

                # C. 제목 및 상세 URL
                # sbj 내부에서 브랜드 링크가 아닌 진짜 제목 링크 선택
                title_link = item.select_one(".sbj a:not(.txt_brand)")
                if title_link:
                    title = title_link.get('title') or title_link.get_text(strip=True)
                    product_url = title_link.get('href')
                else:
                    continue

                # D. 가격 (숫자만 남기기 위해 정제 로직 추가)
                price_tag = item.select_one(".td_prc .prc strong")
                price_raw = price_tag.get_text(strip=True) if price_tag else "0"
                # '2,200円' -> '2200'
                price_numeric = price_raw.replace('円', '').replace(',', '').strip()

                # E. 평점 (별점 너비%를 5점 만점으로 변환)
                rating_star = item.select_one(".review_rating_star")
                if rating_star and 'style' in rating_star.attrs:
                    width_val = rating_star['style'].split(':')[1].replace('%', '').strip()
                    rating = round(float(width_val) / 20, 2)
                else:
                    rating = 0.0

                # F. 리뷰 개수 (괄호 제거)
                review_tag = item.select_one(".review_total_count")
                reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
                reviews = reviews_raw.replace('(', '').replace(')', '').replace(',', '').strip()

                final_rankings.append({
                    "rank": len(final_rankings) + 1,
                    "brand": brand,
                    "title": title,
                    "rating": rating,
                    "reviews": int(reviews) if reviews.isdigit() else 0,
                    "price_jpy": int(price_numeric) if price_numeric.isdigit() else 0,
                    "url": product_url,
                    "goods_code": goods_code,
                    "platform": "Qoo10",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                print(f"✅ {len(final_rankings)}위: [{brand}] {rating}점 ({reviews}건)")

            except Exception as e:
                continue

        # 3. JSON 저장
        filename = f"qoo10_kbeauty_top10_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(final_rankings, f, ensure_ascii=False, indent=4)
        
        print(f"\n✨ 수집 및 파일 저장 성공: {filename}")
        return final_rankings

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
    finally:
        if driver:
            time.sleep(5)
            driver.quit()

if __name__ == "__main__":
    fetch_qoo10_kbeauty_official_link()

📡 큐텐 공식 검색 페이지 접속 중...
📦 페이지 로드 완료, 데이터 추출 시작...
✅ 1위: [dear,Klairs] 4.65점 (7270건)
✅ 2위: [dear,Klairs] 4.75점 (3266건)
✅ 3위: [dear,Klairs] 4.75점 (1931건)
✅ 4위: [ホリカホリカ] 4.65점 (1880건)
✅ 5위: [DR.GET IT] 4.65점 (1690건)
✅ 6위: [dear,Klairs] 4.65점 (1386건)
✅ 7위: [dear,Klairs] 4.65점 (1076건)
✅ 8위: [dear,Klairs] 4.65점 (929건)
✅ 9위: [ホリカホリカ] 4.75점 (797건)
✅ 10위: [ホリカホリカ] 4.75점 (712건)

✨ 수집 및 파일 저장 성공: qoo10_kbeauty_top10_20260315_214833.json
